In [18]:
import pandas as pd

PRQT_FILE_PATH = r"\\bosch.com\dfsrb\DfsDE\LOC\Rt\BST\09_projects\BMI420\External\02_Product_Development\03_System\04_Accel_System_Development\09_FT_Evaluations\BAI_CA\Possible_slope_correlation_CP3_off\acc_std_mean_CP2_CP3_CP4.parquet"
df_raw = pd.read_parquet(PRQT_FILE_PATH)

# DATA CLEANING AND PREPROC

In [19]:
# Filter Columns
df = df_raw.drop(columns=['prod_norm','prod_idx','lotid_norm', 'test_cod', 'waferid_norm' ])

# Create cold/hot hard_bin and soft_bin columns as target labels
cold = df[df['job_name'].str.contains('COLD', case=False, na=False)].drop_duplicates(
    subset=['wafer_id', 'x_coord', 'y_coord']
)[['wafer_id', 'x_coord', 'y_coord', 'hard_bin', 'soft_bin']].rename(
    columns={'hard_bin': 'cold_hard_bin', 'soft_bin': 'cold_soft_bin'}
)

hot = df[df['job_name'].str.contains('HOT', case=False, na=False)].drop_duplicates(
    subset=['wafer_id', 'x_coord', 'y_coord']
)[['wafer_id', 'x_coord', 'y_coord', 'hard_bin', 'soft_bin']].rename(
    columns={'hard_bin': 'hot_hard_bin', 'soft_bin': 'hot_soft_bin'}
)

df = df.merge(cold, on=['wafer_id', 'x_coord', 'y_coord'], how='left')
df = df.merge(hot, on=['wafer_id', 'x_coord', 'y_coord'], how='left')
df[['wafer_id', 'x_coord', 'y_coord', 'job_name', 'hard_bin', 'soft_bin',
    'cold_hard_bin', 'cold_soft_bin', 'hot_hard_bin', 'hot_soft_bin']].head(20)

# Pivot test_txt → test_result for CP2 rows only
cp2 = df[df['job_name'] == 'Herschel_CA_CP2_V2']
pivoted = cp2.pivot_table(
    index=['wafer_id', 'x_coord', 'y_coord'],
    columns='test_txt',
    values='test_result',
    aggfunc='first'
).reset_index()

# Merge in cold/hot bin columns (one row per die)
bins = df[['wafer_id', 'x_coord', 'y_coord',
           'cold_hard_bin', 'cold_soft_bin',
           'hot_hard_bin', 'hot_soft_bin']].drop_duplicates()

pivoted = pivoted.merge(bins, on=['wafer_id', 'x_coord', 'y_coord'], how='left')

# part_fail: True if either hard_bin or soft_bin is not 1
pivoted['cold_part_fail'] = ~((pivoted['cold_hard_bin'] == 1) & (pivoted['cold_soft_bin'] == 1))
pivoted['hot_part_fail'] = ~((pivoted['hot_hard_bin'] == 1) & (pivoted['hot_soft_bin'] == 1))
pivoted

,wafer_id,x_coord,y_coord,T17_62_FW_Combo_Sense_CBIST:F0_ACCALPN_MEAN_X[1],T17_62_FW_Combo_Sense_CBIST:F0_ACCALPN_MEAN_Y[1],T17_62_FW_Combo_Sense_CBIST:F0_ACCALPN_MEAN_Z[1],T17_62_FW_Combo_Sense_CBIST:F0_ACCALPN_SD_X[1],T17_62_FW_Combo_Sense_CBIST:F0_ACCALPN_SD_Y[1],T17_62_FW_Combo_Sense_CBIST:F0_ACCALPN_SD_Z[1],T17_62_FW_Combo_Sense_CBIST:F0_ACCGMN_MEAN_X[1],...,T17_65_FW_Combo_Noise_Remeas:ACCGMN_F7_HPM_3HOT_MEAN_Z[1],T17_65_FW_Combo_Noise_Remeas:ACCGMN_F7_HPM_3HOT_SD_X[1],T17_65_FW_Combo_Noise_Remeas:ACCGMN_F7_HPM_3HOT_SD_Y[1],T17_65_FW_Combo_Noise_Remeas:ACCGMN_F7_HPM_3HOT_SD_Z[1],cold_hard_bin,cold_soft_bin,hot_hard_bin,hot_soft_bin,cold_part_fail,hot_part_fail
0,DPK456-11-C0,1,50,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,20.0,2033.0,20.0,2033.0,True,True
1,DPK456-11-C0,1,51,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,20.0,2033.0,20.0,2033.0,True,True
2,DPK456-11-C0,1,52,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,20.0,2033.0,20.0,2033.0,True,True
3,DPK456-11-C0,1,53,1102.0,694.0,-294.0,266.0,271.0,256.0,18.0,...,-1930.0,602.0,699.0,571.0,1.0,1.0,1.0,1.0,False,False
4,DPK456-11-C0,1,54,1249.0,-1068.0,-382.0,270.0,281.0,258.0,182.0,...,-2310.0,642.0,632.0,619.0,1.0,1.0,1.0,1.0,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51562,DPK456-17-B4,92,57,-375.0,-327.0,-255.0,277.0,264.0,263.0,-428.0,...,-1683.0,631.0,555.0,639.0,1.0,1.0,1.0,1.0,False,False
51563,DPK456-17-B4,92,58,-750.0,-1077.0,-1839.0,266.0,267.0,266.0,-708.0,...,-7217.0,666.0,623.0,585.0,1.0,1.0,1.0,1.0,False,False
51564,DPK456-17-B4,92,59,-775.0,-779.0,-868.0,267.0,278.0,279.0,-680.0,...,-3859.0,572.0,576.0,635.0,1.0,1.0,1.0,1.0,False,False
51565,DPK456-17-B4,92,60,-1595.0,62.0,-1250.0,270.0,297.0,263.0,-925.0,...,-5722.0,622.0,733.0,514.0,1.0,1.0,1.0,1.0,False,False


# PREDICTION

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, f1_score, precision_recall_curve
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
import numpy as np
import re

THRESHOLD = 0.5  # Lower = more recall (catch more fails), less precision

targets = ['hot_part_fail', 'cold_part_fail']
features = [col for col in pivoted.columns if col not in ['wafer_id', 'cold_hard_bin', 'cold_soft_bin', 'hot_hard_bin', 'hot_soft_bin'] + targets]

# Drop rows with NaN in features or targets
data = pivoted.dropna(subset=features + targets).copy()

# Convert bin columns to categorical (nominal values, not numeric)
for t in targets:
    data[t] = data[t].astype(int).astype(str)

# Sanitize feature names for XGBoost (no [, ], or <)
clean_names = {col: re.sub(r'[\[\]<]', '_', col) for col in features}
data = data.rename(columns=clean_names)
features_clean = [clean_names[f] for f in features]

X = data[features_clean].values
groups = data['wafer_id'].values  # group by wafer to prevent leakage

# Store results for dashboard
results = {}

# Split by wafer_id so no wafer appears in both train and test
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for target in targets:
    print(f"\n{'='*60}")
    print(f"Target: {target} (threshold={THRESHOLD})")
    print(f"{'='*60}")

    counts = data[target].value_counts()
    valid_classes = counts[counts >= 2].index
    mask = data[target].isin(valid_classes).values
    X_filtered = X[mask]
    y_raw = data[target].values[mask]
    groups_filtered = groups[mask]

    le = LabelEncoder()
    y = le.fit_transform(y_raw)

    # Identify the "fail" class index (the one encoded from "1" i.e. True)
    fail_class_idx = list(le.classes_).index('1')

    # GroupShuffleSplit: entire wafers go to train or test, never both
    train_idx, test_idx = next(gss.split(X_filtered, y, groups=groups_filtered))
    X_train, X_test = X_filtered[train_idx], X_filtered[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    n_train_wafers = len(set(groups_filtered[train_idx]))
    n_test_wafers = len(set(groups_filtered[test_idx]))
    print(f"Train: {len(X_train)} dies from {n_train_wafers} wafers | Test: {len(X_test)} dies from {n_test_wafers} wafers")

    sample_weights = compute_sample_weight('balanced', y_train)

    model = XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        eval_metric='mlogloss',
    )
    model.fit(X_train, y_train, sample_weight=sample_weights)

    # Use custom threshold on fail probability
    y_proba = model.predict_proba(X_test)[:, fail_class_idx]
    y_pred = (y_proba >= THRESHOLD).astype(int)
    # Map back: 1 = fail_class_idx, 0 = other
    if fail_class_idx == 1:
        pass  # already correct: 1=fail, 0=pass
    else:
        y_pred = 1 - y_pred  # flip if fail is class 0

    present_labels = sorted(set(y_test) | set(y_pred))
    present_names = [str(le.classes_[i]) for i in present_labels]

    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro', labels=present_labels)
    f1_weighted = f1_score(y_test, y_pred, average='weighted', labels=present_labels)
    cm = confusion_matrix(y_test, y_pred, labels=present_labels)
    report = classification_report(y_test, y_pred, labels=present_labels,
                                   target_names=present_names, output_dict=True)

    # Precision-recall curve for the fail class
    pr_precision, pr_recall, pr_thresholds = precision_recall_curve(y_test, y_proba, pos_label=fail_class_idx)

    results[target] = {
        'model': model,
        'le': le,
        'accuracy': acc,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'confusion_matrix': cm,
        'class_names': present_names,
        'report': report,
        'feature_importances': model.feature_importances_,
        'pr_precision': pr_precision,
        'pr_recall': pr_recall,
        'pr_thresholds': pr_thresholds,
        'threshold': THRESHOLD,
    }

    print(f"Accuracy: {acc:.4f} | F1 macro: {f1_macro:.4f} | F1 weighted: {f1_weighted:.4f}")
    print(classification_report(y_test, y_pred, labels=present_labels, target_names=present_names))

print(f"\nAll models trained with threshold={THRESHOLD}. Split by wafer_id (no leakage).")


Target: hot_part_fail (threshold=0.5)
Accuracy: 0.9334 | F1 macro: 0.6421 | F1 weighted: 0.9394
              precision    recall  f1-score   support

           0       0.97      0.96      0.96      9906
           1       0.27      0.39      0.32       408

    accuracy                           0.93     10314
   macro avg       0.62      0.68      0.64     10314
weighted avg       0.95      0.93      0.94     10314


Target: cold_part_fail (threshold=0.5)
Accuracy: 0.9335 | F1 macro: 0.6546 | F1 weighted: 0.9384
              precision    recall  f1-score   support

           0       0.97      0.96      0.96      9873
           1       0.30      0.41      0.34       441

    accuracy                           0.93     10314
   macro avg       0.64      0.68      0.65     10314
weighted avg       0.94      0.93      0.94     10314


All models trained with threshold=0.5. Results stored in `results` dict.


In [31]:
from dash import Dash, html, dcc, Input, Output
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd

app = Dash(__name__)

# --- Precompute figures ---

# 1) Overview bar chart
overview_df = pd.DataFrame([
    {'Target': t, 'Accuracy': r['accuracy'], 'F1 Macro': r['f1_macro'], 'F1 Weighted': r['f1_weighted']}
    for t, r in results.items()
])

fig_overview = go.Figure()
for metric, color in [('Accuracy', '#636EFA'), ('F1 Macro', '#EF553B'), ('F1 Weighted', '#00CC96')]:
    fig_overview.add_trace(go.Bar(
        name=metric, x=overview_df['Target'], y=overview_df[metric],
        text=overview_df[metric].round(3), textposition='outside',
        marker_color=color
    ))
fig_overview.update_layout(
    barmode='group', title='Model Performance Overview',
    yaxis=dict(range=[0, 1.1], title='Score'), xaxis_title='Target',
    template='plotly_white', height=400, legend=dict(orientation='h', y=1.12)
)

# 2) Confusion matrices
cm_figs = {}
for t, r in results.items():
    cm = r['confusion_matrix']
    names = r['class_names']
    fig_cm = go.Figure(data=go.Heatmap(
        z=cm, x=names, y=names,
        colorscale='Blues', text=cm, texttemplate='%{text}',
        hovertemplate='True: %{y}<br>Predicted: %{x}<br>Count: %{z}<extra></extra>'
    ))
    fig_cm.update_layout(
        title=f'Confusion Matrix — {t}', xaxis_title='Predicted', yaxis_title='Actual',
        template='plotly_white', height=450, yaxis=dict(autorange='reversed')
    )
    cm_figs[t] = fig_cm

# 3) Feature importance (top 20 per target)
fi_figs = {}
for t, r in results.items():
    imp = r['feature_importances']
    top_idx = np.argsort(imp)[-20:]
    fi_df = pd.DataFrame({
        'Feature': [features_clean[i] for i in top_idx],
        'Importance': imp[top_idx]
    }).sort_values('Importance')
    fig_fi = px.bar(fi_df, x='Importance', y='Feature', orientation='h',
                    color='Importance', color_continuous_scale='Viridis')
    fig_fi.update_layout(
        title=f'Top 20 Feature Importances — {t}',
        template='plotly_white', height=500, showlegend=False
    )
    fi_figs[t] = fig_fi

# 4) Precision-Recall curves
pr_figs = {}
for t, r in results.items():
    fig_pr = go.Figure()
    fig_pr.add_trace(go.Scatter(
        x=r['pr_recall'], y=r['pr_precision'],
        mode='lines', line=dict(color='#636EFA', width=2),
        name='PR Curve'
    ))
    # Mark the chosen threshold
    th = r['threshold']
    th_idx = np.argmin(np.abs(r['pr_thresholds'] - th))
    fig_pr.add_trace(go.Scatter(
        x=[r['pr_recall'][th_idx]], y=[r['pr_precision'][th_idx]],
        mode='markers', marker=dict(size=14, color='red', symbol='x'),
        name=f'Threshold={th}'
    ))
    fig_pr.update_layout(
        title=f'Precision-Recall Curve (Fail class) — {t}',
        xaxis_title='Recall', yaxis_title='Precision',
        template='plotly_white', height=400,
        xaxis=dict(range=[0, 1.05]), yaxis=dict(range=[0, 1.05]),
    )
    pr_figs[t] = fig_pr

# 5) Per-class metrics tables
def make_report_table(report):
    rows = []
    for cls, metrics in report.items():
        if cls in ('accuracy', 'macro avg', 'weighted avg'):
            continue
        rows.append({
            'Class': cls,
            'Precision': f"{metrics['precision']:.3f}",
            'Recall': f"{metrics['recall']:.3f}",
            'F1-Score': f"{metrics['f1-score']:.3f}",
            'Support': int(metrics['support']),
        })
    for avg in ('macro avg', 'weighted avg'):
        if avg in report:
            m = report[avg]
            rows.append({
                'Class': avg,
                'Precision': f"{m['precision']:.3f}",
                'Recall': f"{m['recall']:.3f}",
                'F1-Score': f"{m['f1-score']:.3f}",
                'Support': int(m['support']),
            })
    return rows

target_options = [{'label': t, 'value': t} for t in targets]

app.layout = html.Div(style={'fontFamily': 'Segoe UI, Arial, sans-serif', 'padding': '20px',
                              'backgroundColor': '#f8f9fa'}, children=[
    html.H1('XGBoost Model Performance Report',
            style={'textAlign': 'center', 'color': '#2c3e50', 'marginBottom': '5px'}),
    html.P(f'Predicting cold/hot part_fail from CP2 test features | Threshold = {THRESHOLD}',
           style={'textAlign': 'center', 'color': '#7f8c8d', 'marginBottom': '30px'}),

    # Overview section
    html.Div(style={'backgroundColor': 'white', 'borderRadius': '10px', 'padding': '20px',
                     'boxShadow': '0 2px 8px rgba(0,0,0,0.1)', 'marginBottom': '25px'}, children=[
        dcc.Graph(figure=fig_overview)
    ]),

    # Target selector
    html.Div(style={'backgroundColor': 'white', 'borderRadius': '10px', 'padding': '20px',
                     'boxShadow': '0 2px 8px rgba(0,0,0,0.1)', 'marginBottom': '25px'}, children=[
        html.Label('Select Target:', style={'fontWeight': 'bold', 'fontSize': '16px', 'marginBottom': '8px'}),
        dcc.Dropdown(id='target-select', options=target_options, value=targets[0],
                     clearable=False, style={'width': '350px'}),
    ]),

    # Confusion matrix + classification report side by side
    html.Div(style={'display': 'flex', 'gap': '20px', 'marginBottom': '25px'}, children=[
        html.Div(style={'flex': '1', 'backgroundColor': 'white', 'borderRadius': '10px', 'padding': '20px',
                         'boxShadow': '0 2px 8px rgba(0,0,0,0.1)'}, children=[
            dcc.Graph(id='confusion-matrix')
        ]),
        html.Div(style={'flex': '1', 'backgroundColor': 'white', 'borderRadius': '10px', 'padding': '20px',
                         'boxShadow': '0 2px 8px rgba(0,0,0,0.1)'}, children=[
            html.H3('Classification Report', style={'color': '#2c3e50', 'marginTop': '0'}),
            html.Div(id='report-table')
        ]),
    ]),

    # Precision-Recall curve
    html.Div(style={'backgroundColor': 'white', 'borderRadius': '10px', 'padding': '20px',
                     'boxShadow': '0 2px 8px rgba(0,0,0,0.1)', 'marginBottom': '25px'}, children=[
        dcc.Graph(id='pr-curve')
    ]),

    # Feature importance
    html.Div(style={'backgroundColor': 'white', 'borderRadius': '10px', 'padding': '20px',
                     'boxShadow': '0 2px 8px rgba(0,0,0,0.1)'}, children=[
        dcc.Graph(id='feature-importance')
    ]),
])

@app.callback(
    Output('confusion-matrix', 'figure'),
    Output('feature-importance', 'figure'),
    Output('report-table', 'children'),
    Output('pr-curve', 'figure'),
    Input('target-select', 'value'),
)
def update_dashboard(selected_target):
    cm_fig = cm_figs[selected_target]
    fi_fig = fi_figs[selected_target]
    pr_fig = pr_figs[selected_target]

    rows = make_report_table(results[selected_target]['report'])
    header = html.Tr([html.Th(c, style={'padding': '10px 14px', 'borderBottom': '2px solid #dee2e6',
                                         'backgroundColor': '#f1f3f5', 'color': '#495057'})
                       for c in ['Class', 'Precision', 'Recall', 'F1-Score', 'Support']])
    body = [
        html.Tr([
            html.Td(row[c], style={
                'padding': '8px 14px', 'borderBottom': '1px solid #eee',
                'fontWeight': 'bold' if row['Class'] in ('macro avg', 'weighted avg') else 'normal',
                'backgroundColor': '#f8f9fa' if row['Class'] in ('macro avg', 'weighted avg') else 'white',
            }) for c in ['Class', 'Precision', 'Recall', 'F1-Score', 'Support']
        ]) for row in rows
    ]
    table = html.Table([html.Thead(header), html.Tbody(body)],
                       style={'width': '100%', 'borderCollapse': 'collapse', 'fontSize': '14px'})

    return cm_fig, fi_fig, table, pr_fig

app.run(jupyter_mode='inline', debug=False, port=8051)